In [1]:
import findspark
findspark.init()

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.appName("windowcode").master('local[3]').config('spark.driver.host','127.0.0.1')\
.config('spark.ui.port',0).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/29 04:53:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
spark

In [9]:
from pyspark.sql.types import StringType,StructField,StructType,IntegerType

In [10]:
sch = StructType([StructField('name',StringType(),True),\
                 StructField('department',StringType(),True),\
                 StructField('salary',IntegerType(),True)])

In [11]:
df = spark.read.format('csv').schema(sch).load(r'/home/AzureAdmin/data/windowdata.txt')

In [12]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)



In [13]:
df.show()

+-------+----------+------+
|   name|department|salary|
+-------+----------+------+
|  James|     Sales|  3000|
|Michael|     Sales|  4600|
| Robert|     Sales|  4100|
|  Maria|   Finance|  3000|
|    Ram|     Sales|  3000|
|  Scott|   Finance|  3300|
|    Jen|   Finance|  3900|
|   Jeff| Marketing|  3000|
|  Kumar| Marketing|  2000|
| Sundar|     Sales|  4100|
|   Amit|     Sales|  4600|
+-------+----------+------+



In [14]:
df.groupby('department').max('salary').show()

+----------+-----------+
|department|max(salary)|
+----------+-----------+
|     Sales|       4600|
|   Finance|       3900|
| Marketing|       3000|
+----------+-----------+



In [17]:
from pyspark.sql.window import *
from pyspark.sql.functions import *

In [16]:
win1 = Window.partitionBy('department')

In [18]:
res = df.withColumn('maxsal',max('salary').over(win1))

In [19]:
res.show()

+-------+----------+------+------+
|   name|department|salary|maxsal|
+-------+----------+------+------+
|  Maria|   Finance|  3000|  3900|
|  Scott|   Finance|  3300|  3900|
|    Jen|   Finance|  3900|  3900|
|   Jeff| Marketing|  3000|  3000|
|  Kumar| Marketing|  2000|  3000|
|  James|     Sales|  3000|  4600|
|Michael|     Sales|  4600|  4600|
| Robert|     Sales|  4100|  4600|
|    Ram|     Sales|  3000|  4600|
| Sundar|     Sales|  4100|  4600|
|   Amit|     Sales|  4600|  4600|
+-------+----------+------+------+



In [20]:
res = df.withColumn('diffsal',max('salary').over(win1)- col('salary'))
res.show()

+-------+----------+------+-------+
|   name|department|salary|diffsal|
+-------+----------+------+-------+
|  Maria|   Finance|  3000|    900|
|  Scott|   Finance|  3300|    600|
|    Jen|   Finance|  3900|      0|
|   Jeff| Marketing|  3000|      0|
|  Kumar| Marketing|  2000|   1000|
|  James|     Sales|  3000|   1600|
|Michael|     Sales|  4600|      0|
| Robert|     Sales|  4100|    500|
|    Ram|     Sales|  3000|   1600|
| Sundar|     Sales|  4100|    500|
|   Amit|     Sales|  4600|      0|
+-------+----------+------+-------+



In [23]:
win2 = Window.partitionBy('department').orderBy('salary')

In [24]:
rnum = df.withColumn('rownumber',row_number().over(win2))
rnum.show()

+-------+----------+------+---------+
|   name|department|salary|rownumber|
+-------+----------+------+---------+
|  Maria|   Finance|  3000|        1|
|  Scott|   Finance|  3300|        2|
|    Jen|   Finance|  3900|        3|
|  Kumar| Marketing|  2000|        1|
|   Jeff| Marketing|  3000|        2|
|  James|     Sales|  3000|        1|
|    Ram|     Sales|  3000|        2|
| Robert|     Sales|  4100|        3|
| Sundar|     Sales|  4100|        4|
|Michael|     Sales|  4600|        5|
|   Amit|     Sales|  4600|        6|
+-------+----------+------+---------+



In [25]:
rnum = df.withColumn('rank',rank().over(win2))
rnum.show()

+-------+----------+------+----+
|   name|department|salary|rank|
+-------+----------+------+----+
|  Maria|   Finance|  3000|   1|
|  Scott|   Finance|  3300|   2|
|    Jen|   Finance|  3900|   3|
|  Kumar| Marketing|  2000|   1|
|   Jeff| Marketing|  3000|   2|
|  James|     Sales|  3000|   1|
|    Ram|     Sales|  3000|   1|
| Robert|     Sales|  4100|   3|
| Sundar|     Sales|  4100|   3|
|Michael|     Sales|  4600|   5|
|   Amit|     Sales|  4600|   5|
+-------+----------+------+----+



In [26]:
rnum = df.withColumn('drank',dense_rank().over(win2))
rnum.show()

+-------+----------+------+-----+
|   name|department|salary|drank|
+-------+----------+------+-----+
|  Maria|   Finance|  3000|    1|
|  Scott|   Finance|  3300|    2|
|    Jen|   Finance|  3900|    3|
|  Kumar| Marketing|  2000|    1|
|   Jeff| Marketing|  3000|    2|
|  James|     Sales|  3000|    1|
|    Ram|     Sales|  3000|    1|
| Robert|     Sales|  4100|    2|
| Sundar|     Sales|  4100|    2|
|Michael|     Sales|  4600|    3|
|   Amit|     Sales|  4600|    3|
+-------+----------+------+-----+



In [27]:
rnum = df.withColumn('lead',lead('salary').over(win2))
rnum.show()

+-------+----------+------+----+
|   name|department|salary|lead|
+-------+----------+------+----+
|  Maria|   Finance|  3000|3300|
|  Scott|   Finance|  3300|3900|
|    Jen|   Finance|  3900|NULL|
|  Kumar| Marketing|  2000|3000|
|   Jeff| Marketing|  3000|NULL|
|  James|     Sales|  3000|3000|
|    Ram|     Sales|  3000|4100|
| Robert|     Sales|  4100|4100|
| Sundar|     Sales|  4100|4600|
|Michael|     Sales|  4600|4600|
|   Amit|     Sales|  4600|NULL|
+-------+----------+------+----+

